# DB증권 Open API — 국내주식 트레이딩 기초 (kr-stock-trading-basic)

`dbsec_sdk`(비동기 클라이언트)로 **국내주식 매매의 기본 흐름을 처음부터 끝까지** 실습하는 노트북입니다 —
시세·차트 조회부터 계좌 확인, 주문(시장가/지정가), 체결 조회, 정정·취소, 동시 조회, 실시간 수신까지
한 번에 따라 할 수 있습니다.

**다루는 내용**

| # | 섹션 | API Endpoint | TR |
|---|---|---|---|
| 1 | 셋업 — 비동기 클라이언트 생성 | - | - |
| 2 | 접근토큰 (발급/캐시 · auto_token) | `/oauth2/token` | token |
| 3 | 시세 — 현재가 / 호가 | `/api/v1/quote/kr-stock/inquiry/price`<br>`/api/v1/quote/kr-stock/inquiry/orderbook` | PRICE<br>HOGA |
| 4 | 차트 — 분차트 | `/api/v1/quote/kr-chart/min` | CHARTMIN |
| 5 | 계좌 — 잔고 / 예수금 | `/api/v1/trading/kr-stock/inquiry/balance`<br>`/api/v1/trading/kr-stock/inquiry/acnt-deposit` | CSPAQ03420<br>CDPCQ00100 |
| 6 | 주문가능수량 | `/api/v1/trading/kr-stock/inquiry/able-orderqty` | CSPBQ00100 |
| 7 | 주문 — 시장가 / 지정가 매수 | `/api/v1/trading/kr-stock/order` | CSPAT00600 |
| 8 | 체결/미체결 조회 | `/api/v1/trading/kr-stock/inquiry/transaction-history` | CSPAQ04800 |
| 9 | 정정 / 취소 | `/api/v1/trading/kr-stock/order-revision`<br>`/api/v1/trading/kr-stock/order-cancel` | CSPAT00700<br>CSPAT00800 |
| 10 | 동시 조회 (`asyncio.gather`) | - | - |
| 11 | 실시간 WebSocket (참고) | `wss://openapi.dbsec.co.kr:7070/websocket` | S00 |

> ⚠️ **주문(7·9번) 셀은 실제 매매가 실행됩니다.** 기본값은 `RUN_ORDER_EXAMPLES = False` 라서
> 그냥 전체 실행(Run All)해도 주문은 **스킵**됩니다. 주문까지 실습하려면 반드시
> **모의투자(`config.yaml` 의 `mode: "demo"`)** 로 바꾼 뒤 플래그를 `True` 로 변경하세요.

**사전 준비** (레포 루트에서)

```bash
pip install -e .            # dbsec_sdk 설치 (requests, pyyaml, websockets 등 포함)
pip install pandas jupyter  # DataFrame 출력 + 노트북 실행
cp config.yaml.example config.yaml   # 앱키 입력 + mode: demo
```

## 1. 셋업 — 비동기 클라이언트 생성

`DBSecClient` 는 **비동기** 클라이언트입니다. 노트북(IPython)은 셀에서 **top-level `await`** 를
지원하므로 `asyncio.run()` 없이 바로 `await` 하면 됩니다.

클라이언트가 처리해 주는 것:
- **유량제어(선제)**: 앱 20TPS + 엔드포인트별 TPS 2-tier 자동 페이싱 (`IGW00201` 사전 차단)
- **유량제어(반응형)**: 그래도 `IGW00201` 이 오면 **지수백오프(1·2·4·8초) 재시도**로 회복
- **토큰 자동발급(`auto_token`, 기본 True)**: 토큰이 없거나 무효/만료(`IGW00121`/`00123`)면 자동 발급·재발급(stdin 프롬프트 없음)

> 아래 셀에서 유량제어·토큰 옵션을 **토글**로 노출했습니다 — `rate_limit_safety`(선제 페이싱 강도, 기본 `0.9`),
> `rate_limit_backoff`(IGW00201 지수백오프 재시도, 기본 `True`), `auto_token`(토큰 자동 발급/재발급, 기본 `True`).
> 운영 권장은 **0.9 + 백오프 ON** (실측: safety 1.0=20TPS지만 호출초과↑, 0.9=18TPS·여유, 0.8=16TPS·보수적).

In [ ]:
import asyncio, sys, pathlib

# 레포 루트를 import 경로에 추가 (노트북이 dbsec_sdk/client_examples/ 안에 있으므로 위로 올라가며 탐색)
ROOT = pathlib.Path.cwd()
while not (ROOT / "dbsec_sdk").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "dbsec_sdk").exists(), "레포 루트를 찾지 못했습니다 — 노트북을 dbsec_sdk/client_examples/ 폴더에서 여세요."
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.unicode.east_asian_width", True)   # 한글 컬럼 정렬

from dbsec_sdk import DBSecClient

client = DBSecClient(
    str(ROOT / "config.yaml"),
    # ── 유량제어 / 토큰 옵션 (기본값을 명시적으로 노출 — 필요 시 토글) ──
    rate_limit=True,          # 선제 페이싱: 호출 전 TPS 간격을 맞춰 IGW00201(호출초과)을 예방. False면 페이싱 끔.
    rate_limit_safety=0.9,    # 안전계수(0<s≤1): 실제율 = 서버TPS × s.  0.9=90%(10% 여유, 권장) /
                              # 1.0=마진0(처리율 최대지만 순간버스트로 IGW00201↑) / 0.8=보수적(안전, 처리율↓)
    rate_limit_backoff=True,  # 지수백오프 재시도: IGW00201 받으면 1·2·4·8초 후 재시도(선제 페이싱이 놓친 순간버스트를 사후 회복).
                              # False면 IGW00201을 즉시 APIError로 표면화.
    auto_token=True,          # 토큰 자동 발급/재발급: True면 요청 시 토큰이 없거나 만료/무효(IGW00121/123)면 자동 발급·재발급(프롬프트 없음).
                              # False면 자동 발급 안 함 → 토큰 없으면 AuthError. get_token()/force_refresh()/revoke()로 직접 관리.
)
print("mode =", client.config.mode, " (demo=모의투자 / production=실전)")
print(f"유량제어: rate_limit={client._core._rate.enabled} safety={client._core._rate.safety} backoff={client._core._rate_limit_backoff} | auto_token={client._core._auto_token}")

In [ ]:
RUN_ORDER_EXAMPLES = True   # ⚠️ True 로 바꾸면 7·9번 주문 셀이 "실제로" 실행됩니다 (모의투자 demo 권장!)

def order_guard() -> bool:
    """주문 셀 공통 가드 — 기본은 스킵, production 모드면 강한 경고."""
    if not RUN_ORDER_EXAMPLES:
        print("⏭ 주문 예제 스킵 — 실행하려면 위 셀의 RUN_ORDER_EXAMPLES = True 로 변경하세요.")
        return False
    if client.config.mode != "demo":
        print("⚠️ 경고: production(실전) 모드입니다 — 아래 주문은 실제 매매로 체결될 수 있습니다!")
    return True

def order_result(resp, *, label="주문"):
    """주문/정정/취소 응답을 안전하게 해석한다.

    장마감·거부 등으로 주문이 접수되지 않으면 서버는 HTTP 200 을 주면서도
    rsp_cd != '00000' 이고 'Out' 블록을 내려주지 않는다. 이때 resp.body['Out']['OrdNo'] 를
    곧바로 인덱싱하면 KeyError 가 나므로, 반드시 이 헬퍼로 감싸 해석한다.

    반환: 접수 성공 시 주문번호(int — OrgOrdNo 로 정정/취소에 재사용) · 실패/미접수면 None.
    """
    ord_no = (resp.body.get("Out") or {}).get("OrdNo")
    if resp.rsp_cd == "00000" and ord_no not in (None, ""):
        print(f"✅ {label} 접수 완료 — 주문번호 {ord_no}  [{resp.rsp_cd}]")
        try:
            return int(ord_no)                 # OrgOrdNo(int) 로 재사용
        except (TypeError, ValueError):
            return ord_no
    if resp.rsp_cd == "00000":                 # 성공이나 주문번호 없음(예: 일부 취소 응답)
        print(f"✅ {label} 처리 완료 — [{resp.rsp_cd}] {resp.rsp_msg}")
        return None
    print(f"⛔ {label} 미접수 — [{resp.rsp_cd}] {resp.rsp_msg}")
    print("   (장마감·거부 등으로 접수되지 않았습니다 — 주문번호 없음. 에러가 아니라 정상 처리입니다.)")
    return None

## 2. 접근토큰 — 발급 / 캐시 (auto_token)

토큰은 24시간 유효하며 저장소 루트의 `.dbsec_token.json` 에 캐시됩니다(`examples/` 스크립트와 공유).
유효한 캐시가 있으면 재발급 없이 그대로 사용하고, 없으면 `auto_token=True`(기본) 시 자동 발급합니다(stdin 프롬프트 없음).
토큰을 직접 다루려면 `await client.get_token()` / `force_refresh()` / `revoke()` 를 쓰고, `auto_token=False` 면 발급은 전적으로 수동입니다.

In [ ]:
token = await client.get_token()
print("토큰 확보 — 앞 16자:", token[:16] + "...  (루트 .dbsec_token.json 에 캐시)")

## 3. 시세 — 현재가 / 호가 (삼성전자)

- `InputCondMrktDivCode`: `J` KRX주식 · `NJ` NXT · `UJ` 통합 · `EN` ETN · `W` ELW · `U` 업종지수
- 응답은 `APIResponse` — `resp.body`(dict) 그대로 쓰거나 `resp.to_dataframe()` 으로 변환

In [ ]:
# 현재가조회 (PRICE, 5 TPS)
resp = await client.apis.kr_stock_quote.kr_stock_inquire_price(
    InputCondMrktDivCode="J",   # KRX 주식
    InputIscd1="005930",        # 삼성전자
)
last_price = int(resp.body["Out"]["Prpr"])   # 현재가 — 뒤의 지정가 주문 가격 계산에 사용
print("status:", resp.status_code, "/ 현재가:", f"{last_price:,}원")
resp.to_dataframe().T.head(12)

In [ ]:
# 호가조회 (HOGA, 3 TPS) — 매도/매수 10단 호가
resp = await client.apis.kr_stock_quote.kr_stock_inquire_orderbook(
    InputCondMrktDivCode="J",
    InputIscd1="005930")
df = resp.to_dataframe().T
df

## 4. 차트 — 분차트 (10분봉)

`InputDivXtick` 은 초 단위 틱 간격입니다 (600 = 10분봉). 일/주/월차트는
`kr_chart_chart_day` / `kr_chart_chart_week` / `kr_chart_chart_month` 를 사용하세요.

In [ ]:
from datetime import datetime

resp = await client.apis.kr_chart.kr_chart_chart_min(
    InputCondMrktDivCode="J",
    InputIscd1="005930",
    InputDate1=datetime.now().strftime("%Y%m%d"),  # 오늘 (휴장일이면 빈 결과일 수 있음)
    InputDivXtick="600",  # 600초 = 10분봉
    dataCnt="10",
    InputOrgAdjPrc="1")
resp.to_dataframe()

## 4-B. 연속조회 (자동 페이징) — `fetch_all=True`

같은 틱차트 조회를 **① 연속조회 없이(단건)** vs **② `fetch_all=True`(끝까지 자동 병합)** 로 대비합니다.

- **① 단건**: 한 번 호출 = 한 페이지(`dataCnt` 만큼). 데이터가 더 있으면 `has_more=True`/`cont_key` 가 채워지지만 **자동으로 더 가져오지 않습니다**.
- **② `fetch_all=True`**: `cont_yn`/`cont_key` 를 자동 처리하며 끝(`cont_yn='N'`)까지(또는 `max_pages` 까지) 받아 **하나의 `APIResponse` 로 병합**합니다. 원본 페이지는 `resp.pages`.

> 틱차트는 체결이 많아 단건은 `has_more=True`(뒤에 더 있음)가 되고, `fetch_all=True` 는 여러 페이지를 누적합니다.
> 두 셀의 **건수·페이지·`has_more`** 차이를 비교하세요. (틱차트는 방대하니 `fetch_all` 엔 `max_pages` 상한 권장.)

In [ ]:
# ① 연속조회 없이 단건 호출 — 한 번 = 한 페이지(dataCnt 만큼)만 받습니다.
#    has_more=True / cont_key 가 있으면 "뒤에 데이터가 더 있다"는 뜻 (자동으로 안 가져옴 — 아래 ② 와 대비).
from datetime import datetime
day = datetime.now().strftime("%Y%m%d")   # 오늘(장중) 또는 최근 거래일

resp = await client.apis.kr_chart.kr_chart_chart_tick(
    InputCondMrktDivCode="J",    # KRX 주식
    InputIscd1="005930",         # 삼성전자
    InputDate1=day,              # 조회 시작일
    InputDivXtick="1",           # 1틱 단위
    InputOrgAdjPrc="1",          # 수정주가 사용
    dataCnt="100",               # 한 번에 100건
)
n_rows = lambda body: next((len(v) for v in body.values() if isinstance(v, list)), 0)
print(f"[단건] 삼성 틱차트  →  {n_rows(resp.body):,}건 (1페이지)"
      f"  | has_more={resp.has_more}  cont_key={resp.cont_key[:20]!r}")
display(resp.to_dataframe().head(15))

In [ ]:
# ② fetch_all=True — 연속조회로 cont_yn/cont_key 자동 처리하며 끝까지(또는 max_pages 까지) 받아
#    하나의 APIResponse 로 병합합니다. ①의 단건(1페이지)보다 훨씬 많은 데이터를 한 번에.
from datetime import datetime
day = datetime.now().strftime("%Y%m%d")

resp = await client.apis.kr_chart.kr_chart_chart_tick(
    InputCondMrktDivCode="J",    # KRX 주식
    InputIscd1="005930",         # 삼성전자
    InputDate1=day,              # 조회 시작일
    InputDivXtick="1",           # 1틱 단위
    InputOrgAdjPrc="1",          # 수정주가 사용
    dataCnt="100",               # 페이지당 100건
    fetch_all=True,              # ← 연속조회 자동 병합 (이게 ①과의 차이)
    max_pages=10,                # 틱차트는 방대하니 상한 (None/생략 = 끝까지)
)
n_rows = lambda body: next((len(v) for v in body.values() if isinstance(v, list)), 0)
print(f"[fetch_all] 삼성 틱차트  →  페이지 {len(resp.pages)}개 / 누적 {n_rows(resp.body):,}건"
      f"  (마지막 has_more={resp.has_more})")
for i, p in enumerate(resp.pages, 1):    # 원본 페이지들은 resp.pages 로 접근
    print(f"  page{i}: {n_rows(p.body):>4,}건   cont_key={p.cont_key[:20]!r}")
display(resp.to_dataframe().head(15))

## 5. 계좌 — 잔고 / 예수금

In [ ]:
# 주식잔고조회 (CSPAQ03420, 2 TPS) — Out: 계좌 요약, Out1: 보유종목 리스트
resp = await client.apis.kr_stock_order.kr_stock_inquire_balance(QryTpCode0="0")
print("status:", resp.status_code, "| 총평가금액:", resp.body.get("Out", {}).get("TotEvalAmt"))
resp.to_dataframe()   # 보유종목 (없으면 빈 DataFrame)

In [ ]:
# 계좌예수금조회 (CDPCQ00100, 1 TPS)
resp = await client.apis.kr_stock_order.kr_stock_inquire_deposit()
resp.to_dataframe().T.head(12)

## 6. 주문가능수량 — 현재가 기준 매수 가능 수량

In [ ]:
# 주식주문가능수량조회 (CSPBQ00100, 2 TPS)
resp = await client.apis.kr_stock_order.kr_stock_inquire_psbl_quantity(
    BnsTpCode="2",        # 1:매도 2:매수
    IsuNo="A005930",      # A + 6자리 종목코드
    OrdPrc=last_price,    # 현재가 기준
)
resp.to_dataframe().T

## 7. 주문 — 시장가 / 지정가 매수 ⚠️

**여기부터 실제 매매 구간입니다.** `RUN_ORDER_EXAMPLES = False`(기본)면 자동 스킵됩니다.

주요 파라미터 (주식종합주문 CSPAT00600, 10 TPS):

| 파라미터 | 의미 | 값 |
|---|---|---|
| `BnsTpCode` | 매매구분 | `1` 매도 / `2` 매수 |
| `OrdprcPtnCode` | 호가유형 | `00` 지정가 / `03` 시장가 / `05` 조건부지정 / `06` 최유리 / `07` 최우선 |
| `OrdPrc` | 주문가격 | 시장가면 `0` |
| `MgntrnCode` | 신용거래 | `000` 보통(일반주문) · 신용상환 `101`/`103`/`105`/`107`/`180` |
| `LoanDt` | 대출일 | 일반 `"00000000"` · 신용매도는 결제일자(`YYYYMMDD`, 대출일자X) |
| `OrdCndiTpCode` | 주문조건 | `0` 없음 / `1` IOC / `2` FOK |
| `TrchNo` | 거래소 | `1` KRX 고정 |

In [ ]:
# 7-1) 시장가 매수 1주
if order_guard():
    resp = await client.apis.kr_stock_order.kr_stock_order(
        IsuNo="A005930",
        TrchNo=1,
        OrdQty=1,
        OrdPrc=0,
        BnsTpCode="2",
        OrdprcPtnCode="03",  # 03: 시장가
        MgntrnCode="000",
        LoanDt="00000000",
        OrdCndiTpCode="0")
    order_result(resp, label="시장가 매수")   # 장마감·거부 시에도 KeyError 없이 사유만 출력

In [ ]:
# 7-2) 지정가 매수 1주 — 현재가 5% 아래(체결 안 되도록) → 9번 정정/취소 실습용
if order_guard():
    limit_price = int(last_price * 0.95) // 100 * 100   # 100원 단위 절사 (호가단위는 가격대별 상이)
    resp = await client.apis.kr_stock_order.kr_stock_order(
        IsuNo="A005930",
        TrchNo=1,
        OrdQty=1,
        OrdPrc=limit_price,
        BnsTpCode="2",
        OrdprcPtnCode="00",  # 00: 지정가
        MgntrnCode="000",
        LoanDt="00000000",
        OrdCndiTpCode="0")
    limit_ord_no = order_result(resp, label="지정가 매수")   # 미접수(장마감 등)면 None → 9번 셀이 자동 스킵
    if limit_ord_no:
        print(f"   주문가격 {limit_price:,}원 — 9번 정정/취소 실습에서 이 주문번호를 사용합니다.")

## 8. 체결 / 미체결 조회

In [ ]:
# 체결/미체결조회 (CSPAQ04800, 2 TPS)
resp = await client.apis.kr_stock_order.kr_stock_inquire_executions(
    SorTpYn="2",      # 정렬: 2 역순
    ExecYn="0",       # 체결여부: 0 전체 / 1 체결 / 2 미체결
    TrdMktCode="0",   # 시장: 0 전체
    BnsTpCode="0",    # 매매구분: 0 전체
    IsuTpCode="0",    # 종목: 0 전체
    QryTp="0")        # 조회구분: 0 전체
resp.to_dataframe()   # 오늘 주문이 없으면 빈 DataFrame

## 9. 정정 / 취소 — 7-2 지정가 주문 대상 ⚠️

- **정정**(CSPAT00700): 미체결 주문의 가격/수량 변경 — 정정하면 **새 주문번호**가 발급됩니다.
- **취소**(CSPAT00800): 미체결 주문 취소. 이미 체결된 주문은 정정/취소 불가.

In [ ]:
# 9-1) 정정 — 지정가를 4% 아래 가격으로 변경
if order_guard():
    if not globals().get("limit_ord_no"):
        print("정정할 주문이 없습니다 — 7-2 지정가 주문이 접수되지 않았습니다(미실행·장마감·거부 등).")
    else:
        new_price = int(last_price * 0.96) // 100 * 100
        resp = await client.apis.kr_stock_order.kr_stock_order_modify(
            OrgOrdNo=limit_ord_no,
            IsuNo="A005930",
            OrdQty=1,
            OrdprcPtnCode="00",
            OrdCndiTpCode="0",
            OrdPrc=new_price)
        new_no = order_result(resp, label="정정")   # 정정 성공 시 새 주문번호 발급
        if new_no:
            limit_ord_no = new_no                    # 새 주문번호로 갱신
            print(f"   새 가격 {new_price:,}원")

In [ ]:
# 9-2) 취소
if order_guard():
    if not globals().get("limit_ord_no"):
        print("취소할 주문이 없습니다 — 7-2 지정가 주문이 접수되지 않았습니다(미실행·장마감·거부 등).")
    else:
        resp = await client.apis.kr_stock_order.kr_stock_order_cancel(
            OrgOrdNo=limit_ord_no,
            IsuNo="A005930",
            OrdQty=1)
        order_result(resp, label="취소")     # 장마감·거부 시에도 KeyError 없이 사유만 출력
        if resp.rsp_cd == "00000":
            del limit_ord_no                  # 취소 성공 시에만 정리

## 10. 보너스 — `asyncio.gather` 동시 조회

서로 다른 엔드포인트는 각자 유량제어 리미터를 쓰므로 **진짜 병렬**로 끝납니다.
같은 엔드포인트를 여러 번 부르면 그 API 의 TPS 만큼만 동시 진행(자동 페이싱)됩니다.

In [ ]:
import time, json

t = time.monotonic()
price, balance, deposit = await asyncio.gather(
    client.apis.kr_stock_quote.kr_stock_inquire_price(
        InputCondMrktDivCode="J",
        InputIscd1="005930"),
    client.apis.kr_stock_order.kr_stock_inquire_balance(QryTpCode0="0"),
    client.apis.kr_stock_order.kr_stock_inquire_deposit(),
)
print(f"3건 동시 조회 {time.monotonic()-t:.2f}s — status:",
      [r.status_code for r in (price, balance, deposit)])

# 조회 결과를 이름별로 줄바꿈해서 출력 (indent=2 → 보기 좋게 / 한 줄로 보려면 indent=None)
results = {"price": price, "balance": balance, "deposit": deposit}
for name, resp in results.items():
    print(f'\n"{name}" :', json.dumps(resp.body, ensure_ascii=False, indent=2))

## 11. 실시간 WebSocket (참고)

`run()` 이 무한 수신 루프라서 노트북에서는 **커널 Interrupt(■)로 중단**해야 합니다.
필요할 때 `False` 를 `True` 로 바꿔 실행하세요. 그룹별 standalone 예제는
`examples/kr_stock_realtime/` 참고.

**연결이 안 될 때 (트러블슈팅)** — 서버 오류 응답은 로그(warning)로 원인·해결 힌트가 출력됩니다:

| 증상 | 원인 | 해결 |
|---|---|---|
| `rsp_cd=10011` 세션 수 초과 | 이전 세션 미정리 (계좌당 최대 **2세션**) | 아래 세션 초기화 셀 실행 후 재연결 |
| 연결 직후 끊김 | 연결 유량 **1분 6회(6 TPM)** 초과 | 1분 뒤 재시도 |
| `rsp_cd=10004/10006` | tr_cd / tr_key 오입력 | 국내주식 tr_key 는 `"J "` + 종목코드 (J 뒤 공백) |
| `rsp_cd=10009` | 토큰-계좌 불일치 | `await client.force_refresh()` 후 재연결 |

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")  # 서버 응답/오류 힌트 표시

if True:   # 실행하려면 True 로 변경 (중단: 커널 Interrupt)
    # 세션이 막혀 있으면(10011) 먼저 초기화: 접속중인 모든 웹소켓 세션을 정리
    await client.apis.ws_common.ws_session_disconnect()

    ws = client.create_websocket()
    ws.on_message(lambda tr_cd, tr_key, data: print(tr_cd, tr_key, data))
    await ws.connect()
    # tr_type 은 직접 지정: "1"=시세구독 "2"=해제 "3"=계좌등록. 일반 시세는 "1".
    await ws.add_realtime(
        tr_cd="S00",
        tr_key="J 005930",
        tr_type="1")   # 국내주식 체결가 (J + 공백 + 종목코드)
    await ws.run()

---
## 주의사항
> ⚠️ 다시 한 번: 주문 계열 API 는 실제 매매가 실행됩니다. 반드시 **모의투자(demo)** 로 충분히
> 테스트한 뒤 실전(production)으로 전환하세요. MCP 거래·자동매매로 인한 손실 책임은 사용자 본인에게 있습니다.